# 1. Project Title: Traffic Flow Prediction Using Machine Learning

## 2. Project Objective
The primary objective of this project is to build an end-to-end Machine Learning regression pipeline to accurately predict hourly westbound traffic volume along Interstate 94 (I-94) in Minneapolis-St. Paul, Minnesota, utilizing historical traffic sensor counts, meteorological observations, and calendar holiday records.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Core libraries loaded successfully.")

Core libraries loaded successfully.


## 3. Dataset Sources & Acquisition
The datasets originate from the official **UCI Machine Learning Repository: Metro Interstate Traffic Volume Dataset** (DOI: 10.24432/C5X60B). The data includes hourly traffic counts from Minnesota Department of Transportation (MnDOT) ATR station 301, paired with hourly weather observation feeds from Minneapolis-Saint Paul International Airport (MSP).

In [2]:
# Define relative paths to raw datasets
RAW_DIR = "../datasets/raw"

traffic_raw_path = os.path.join(RAW_DIR, "traffic_raw.csv")
weather_raw_path = os.path.join(RAW_DIR, "weather_raw.csv")
holidays_raw_path = os.path.join(RAW_DIR, "holidays_raw.csv")

print(f"Traffic raw path:  {traffic_raw_path} (Exists: {os.path.exists(traffic_raw_path)})")
print(f"Weather raw path:  {weather_raw_path} (Exists: {os.path.exists(weather_raw_path)})")
print(f"Holidays raw path: {holidays_raw_path} (Exists: {os.path.exists(holidays_raw_path)})")

Traffic raw path:  ../datasets/raw\traffic_raw.csv (Exists: True)
Weather raw path:  ../datasets/raw\weather_raw.csv (Exists: True)
Holidays raw path: ../datasets/raw\holidays_raw.csv (Exists: True)


In [3]:
# Load raw datasets
df_traffic = pd.read_csv(traffic_raw_path)
df_weather = pd.read_csv(weather_raw_path)
df_holidays = pd.read_csv(holidays_raw_path)

print("Raw datasets loaded into DataFrames.")

Raw datasets loaded into DataFrames.


## 4. Dataset Dimensions
Inspecting the shape (rows and columns) of each raw data stream.

In [4]:
print(f"Traffic Raw Dimensions:  {df_traffic.shape[0]:,} rows, {df_traffic.shape[1]} columns")
print(f"Weather Raw Dimensions:  {df_weather.shape[0]:,} rows, {df_weather.shape[1]} columns")
print(f"Holidays Raw Dimensions: {df_holidays.shape[0]:,} rows, {df_holidays.shape[1]} columns")

Traffic Raw Dimensions:  48,204 rows, 2 columns
Weather Raw Dimensions:  48,204 rows, 7 columns
Holidays Raw Dimensions: 48,204 rows, 2 columns


## 5. Sample Observations
Displaying the initial 5 records of each ingested dataset.

In [5]:
print("--- TRAFFIC RAW SAMPLE ---")
display(df_traffic.head())

print("--- WEATHER RAW SAMPLE ---")
display(df_weather.head())

print("--- HOLIDAYS RAW SAMPLE ---")
display(df_holidays.head())

--- TRAFFIC RAW SAMPLE ---


,date_time,traffic_volume
0,2012-10-02 09:00:00,5545
1,2012-10-02 10:00:00,4516
2,2012-10-02 11:00:00,4767
3,2012-10-02 12:00:00,5026
4,2012-10-02 13:00:00,4918


--- WEATHER RAW SAMPLE ---


,date_time,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description
0,2012-10-02 09:00:00,288.28,0.0,0.0,40,Clouds,scattered clouds
1,2012-10-02 10:00:00,289.36,0.0,0.0,75,Clouds,broken clouds
2,2012-10-02 11:00:00,289.58,0.0,0.0,90,Clouds,overcast clouds
3,2012-10-02 12:00:00,290.13,0.0,0.0,90,Clouds,overcast clouds
4,2012-10-02 13:00:00,291.14,0.0,0.0,75,Clouds,broken clouds


--- HOLIDAYS RAW SAMPLE ---


,date_time,holiday
0,2012-10-02 09:00:00,NaN
1,2012-10-02 10:00:00,NaN
2,2012-10-02 11:00:00,NaN
3,2012-10-02 12:00:00,NaN
4,2012-10-02 13:00:00,NaN


## 6. Data Types and Schema Inspection

In [6]:
print("--- TRAFFIC SCHEMA ---")
print(df_traffic.dtypes)

print("\n--- WEATHER SCHEMA ---")
print(df_weather.dtypes)

print("\n--- HOLIDAYS SCHEMA ---")
print(df_holidays.dtypes)

--- TRAFFIC SCHEMA ---
date_time           str
traffic_volume    int64
dtype: object

--- WEATHER SCHEMA ---
date_time                  str
temp                   float64
rain_1h                float64
snow_1h                float64
clouds_all               int64
weather_main               str
weather_description        str
dtype: object

--- HOLIDAYS SCHEMA ---
date_time    str
holiday      str
dtype: object


## 7. Missing Value Diagnostics

In [7]:
print("--- TRAFFIC MISSING VALUES ---")
print(df_traffic.isnull().sum())

print("\n--- WEATHER MISSING VALUES ---")
print(df_weather.isnull().sum())

print("\n--- HOLIDAYS MISSING VALUES ---")
print(df_holidays.isnull().sum())
print(f"Note: 'holiday' contains {df_holidays['holiday'].notnull().sum()} marked holidays; NaN denotes standard non-holiday days.")

--- TRAFFIC MISSING VALUES ---
date_time         0
traffic_volume    0
dtype: int64

--- WEATHER MISSING VALUES ---
date_time              0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
dtype: int64

--- HOLIDAYS MISSING VALUES ---
date_time        0
holiday      48143
dtype: int64
Note: 'holiday' contains 61 marked holidays; NaN denotes standard non-holiday days.


## 8. Duplicate Record Diagnostics

In [8]:
print(f"Duplicate rows in Traffic:  {df_traffic.duplicated().sum():,}")
print(f"Duplicate rows in Weather:  {df_weather.duplicated().sum():,}")
print(f"Duplicate rows in Holidays: {df_holidays.duplicated().sum():,}")
print(f"Duplicate timestamps in Traffic: {df_traffic.duplicated(subset=['date_time']).sum():,}")

Duplicate rows in Traffic:  7,629
Duplicate rows in Weather:  17
Duplicate rows in Holidays: 7,629
Duplicate timestamps in Traffic: 7,629


## 9. Descriptive Statistics of Continuous Variables

In [9]:
print("--- TRAFFIC VOLUME SUMMARY ---")
display(df_traffic.describe())

print("--- WEATHER NUMERICAL METRICS SUMMARY ---")
display(df_weather.describe())

--- TRAFFIC VOLUME SUMMARY ---


,traffic_volume
count,48204.000000
mean,3259.818355
std,1986.860670
min,0.000000
25%,1193.000000
50%,3380.000000
75%,4933.000000
max,7280.000000


--- WEATHER NUMERICAL METRICS SUMMARY ---


,temp,rain_1h,snow_1h,clouds_all
count,48204.000000,48204.000000,48204.000000,48204.000000
mean,281.205870,0.334264,0.000222,49.362231
std,13.338232,44.789133,0.008168,39.015750
min,0.000000,0.000000,0.000000,0.000000
25%,272.160000,0.000000,0.000000,1.000000
50%,282.450000,0.000000,0.000000,64.000000
75%,291.806000,0.000000,0.000000,90.000000
max,310.070000,9831.300000,0.510000,100.000000


## 10. Initial Key Observations
1. **Temporal Coverage:** Multi-year hourly observations between 2012 and 2018.
2. **Target Variable (`traffic_volume`):** Ranges from 0 to 7,280 vehicles/hour with a mean around 3,260.
3. **Holiday Feature:** Mostly NaN for standard working days with specific strings for major US holidays.
4. **Weather Telemetry:** `temp` has extreme physical outliers (e.g. 0.0 K dropouts), and `rain_1h` exhibits a spike anomaly (9831 mm) requiring robust cleaning in Notebook 02.